# 

In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import glob
import pickle
import numpy as np
import pandas as pd
import mne


from EEGPrep import finalPreprocesseeg
from EYEPrep import finalPreprocesseye


BASE_PATH = r"E:\koorathota_data\pkl_data\pkl_data\pkl_data"
OUTPUT_DIR = r"E:\koorathota_data\pkl_data\pkl_data\pkl_data\eegdata"

os.makedirs(OUTPUT_DIR, exist_ok=True)

pkl_files = glob.glob(os.path.join(BASE_PATH, "*.pkl"))

all_X_eeg = []
all_y_cls = []
all_y_reg = []
all_subject_id = []
all_session_id = []
all_trial_id = []
all_stats = []

all_X_eye_L = []
all_X_eye_R = []
all_X_eye_M = []

all_y_cls_eye_L = []
all_y_cls_eye_R = []
all_y_cls_eye_M = []

all_y_reg_eye_L = []
all_y_reg_eye_R = []
all_y_reg_eye_M = []

reference_ch_names = None
reference_shape = None
global_trial_counter = 0

print(f"Found {len(pkl_files)} files to process.")



for file_path in pkl_files:
    file_name = os.path.basename(file_path)
    print(f"\nProcessing: {file_name}")

    try:
        with open(file_path, "rb") as f:
            data = pickle.load(f)

        resulteeg = finalPreprocesseeg(data)

        if resulteeg is None:
            raise ValueError(f"Missing required keys in {file_name}")

        (
            raw_ica,
            eeg_epochs,
            filtered_left_peaks,
            filtered_right_peaks,
            filtered_left_onsets,
            filtered_right_onsets,
            events,
            y_class,
            y_reg,
        ) = resulteeg

        resulteye = finalPreprocesseye(data, events)

        if resulteye is None:
            raise ValueError(f"Missing required keys in {file_name}")

        (
            pupil_epochsL,
            epoch_infoL,
            rejectedL,
            rel_timesL,
            kept_indicesL,
            pupil_epochsR,
            epoch_infoR,
            rejectedR,
            rel_timesR,
            kept_indicesR,
            pupil_epochsM,
            epoch_infoM,
            rejectedM,
            rel_timesM,
            kept_indicesM,
        ) = resulteye

        # Fill residual NaNs in pupil epochs
        nan_mask = ~np.isfinite(pupil_epochsL)
        if nan_mask.any():
            print(f"  Filling {nan_mask.sum()} residual NaN samples in left pupil with 0")
        pupil_epochsL[nan_mask] = 0.0

        nan_mask = ~np.isfinite(pupil_epochsR)
        if nan_mask.any():
            print(f"  Filling {nan_mask.sum()} residual NaN samples in right pupil with 0")
        pupil_epochsR[nan_mask] = 0.0

        nan_mask = ~np.isfinite(pupil_epochsM)
        if nan_mask.any():
            print(f"  Filling {nan_mask.sum()} residual NaN samples in mean pupil with 0")
        pupil_epochsM[nan_mask] = 0.0

        # Eye tensors: [N, T, 1]
        X_eye_L = pupil_epochsL[:, :, np.newaxis].astype(np.float32)
        X_eye_R = pupil_epochsR[:, :, np.newaxis].astype(np.float32)
        X_eye_M = pupil_epochsM[:, :, np.newaxis].astype(np.float32)

        # EEG labels
        y_cls_eeg = np.asarray(y_class, dtype=np.int64)
        y_reg_eeg = np.asarray(y_reg, dtype=np.float32)

        # Eye labels aligned to kept indices
        y_cls_eye_L = y_cls_eeg[kept_indicesL]
        y_cls_eye_R = y_cls_eeg[kept_indicesR]
        y_cls_eye_M = y_cls_eeg[kept_indicesM]

        y_reg_eye_L = y_reg_eeg[kept_indicesL]
        y_reg_eye_R = y_reg_eeg[kept_indicesR]
        y_reg_eye_M = y_reg_eeg[kept_indicesM]

        # EEG tensor
        epochs_reve = eeg_epochs.copy().pick("eeg")
        epochs_reve = epochs_reve.copy().resample(200)

        X_eeg = epochs_reve.get_data(copy=True).astype(np.float32)  # [N, C, T]
        ch_names = epochs_reve.ch_names
        n_epochs = X_eeg.shape[0]

        if len(y_cls_eeg) != n_epochs or len(y_reg_eeg) != n_epochs:
            raise ValueError(
                f"Label length mismatch in {file_name}: "
                f"n_epochs={n_epochs}, len(y_cls_eeg)={len(y_cls_eeg)}, len(y_reg_eeg)={len(y_reg_eeg)}"
            )

        if reference_ch_names is None:
            reference_ch_names = ch_names
        elif ch_names != reference_ch_names:
            raise ValueError(f"Channel mismatch in {file_name}")

        if reference_shape is None:
            reference_shape = X_eeg.shape[1:]  # (C, T)
        elif X_eeg.shape[1:] != reference_shape:
            raise ValueError(
                f"Shape mismatch in {file_name}: {X_eeg.shape[1:]} vs {reference_shape}"
            )

        # Subject / session metadata
        info = data["Unity_TrialInfo"]
        info_t = info[0]
        idd = info_t[0]
        session = info_t[1]

        subj_value = idd[0]
        sess_value = session[0]

        subject_id = np.full(n_epochs, subj_value, dtype=np.int64)
        session_id = np.full(n_epochs, sess_value, dtype=np.int64)

        trial_id = np.arange(
            global_trial_counter,
            global_trial_counter + n_epochs,
            dtype=np.int64,
        )
        global_trial_counter += n_epochs

        # Store EEG
        all_X_eeg.append(X_eeg)
        all_y_cls.append(y_cls_eeg)
        all_y_reg.append(y_reg_eeg)
        all_subject_id.append(subject_id)
        all_session_id.append(session_id)
        all_trial_id.append(trial_id)

        # Store Eye
        all_X_eye_L.append(X_eye_L)
        all_X_eye_R.append(X_eye_R)
        all_X_eye_M.append(X_eye_M)

        all_y_cls_eye_L.append(y_cls_eye_L)
        all_y_cls_eye_R.append(y_cls_eye_R)
        all_y_cls_eye_M.append(y_cls_eye_M)

        all_y_reg_eye_L.append(y_reg_eye_L)
        all_y_reg_eye_R.append(y_reg_eye_R)
        all_y_reg_eye_M.append(y_reg_eye_M)

        print(f"  Added {n_epochs} EEG epochs from {file_name}")
        print(f"  Eye kept: L={len(kept_indicesL)}, R={len(kept_indicesR)}, M={len(kept_indicesM)}")

    except Exception as e:
        print(f"  Error processing {file_name}: {e}")

print("\nCombining all files...")

if len(all_X_eeg) == 0:
    raise RuntimeError("No valid files were processed.")

# Combine EEG
X_eeg = np.concatenate(all_X_eeg, axis=0)
y_cls = np.concatenate(all_y_cls, axis=0)
y_reg = np.concatenate(all_y_reg, axis=0)
subject_id = np.concatenate(all_subject_id, axis=0)
session_id = np.concatenate(all_session_id, axis=0)
trial_id = np.concatenate(all_trial_id, axis=0)

# Combine Eye
X_eye_L = np.concatenate(all_X_eye_L, axis=0)
X_eye_R = np.concatenate(all_X_eye_R, axis=0)
X_eye_M = np.concatenate(all_X_eye_M, axis=0)

y_cls_eye_L = np.concatenate(all_y_cls_eye_L, axis=0)
y_cls_eye_R = np.concatenate(all_y_cls_eye_R, axis=0)
y_cls_eye_M = np.concatenate(all_y_cls_eye_M, axis=0)

y_reg_eye_L = np.concatenate(all_y_reg_eye_L, axis=0)
y_reg_eye_R = np.concatenate(all_y_reg_eye_R, axis=0)
y_reg_eye_M = np.concatenate(all_y_reg_eye_M, axis=0)

dataset_path_eeg = os.path.join(OUTPUT_DIR, "reve_eeg_dataset_with_eye.npz")
dataset_path_eye = os.path.join(OUTPUT_DIR, "eye_dataset.npz")

np.savez(
    dataset_path_eeg,
    X_eeg=X_eeg.astype(np.float32),
    y_cls=y_cls.astype(np.int64),
    y_reg=y_reg.astype(np.float32),
    subject_id=subject_id,
    session_id=session_id,
    ch_names=np.asarray(reference_ch_names, dtype=object),
    trial_id=trial_id,
)

np.savez(
    dataset_path_eye,
    X_eye_L=X_eye_L.astype(np.float32),
    X_eye_R=X_eye_R.astype(np.float32),
    X_eye_M=X_eye_M.astype(np.float32),
    y_cls_eye_L=y_cls_eye_L.astype(np.int64),
    y_cls_eye_R=y_cls_eye_R.astype(np.int64),
    y_cls_eye_M=y_cls_eye_M.astype(np.int64),
    y_reg_eye_L=y_reg_eye_L.astype(np.float32),
    y_reg_eye_R=y_reg_eye_R.astype(np.float32),
    y_reg_eye_M=y_reg_eye_M.astype(np.float32),
    subject_id=subject_id,
    session_id=session_id,
    trial_id=trial_id,
)

print("Done.")
print(f"Saved EEG dataset to: {dataset_path_eeg}")
print(f"Saved eye dataset to: {dataset_path_eye}")

C:\Users\USER\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


Found 32 files to process.

Processing: Copy of 08_25_2022_10_32_35-Exp_adadrive-Sbj_12-Ssn_01.dats-025.pkl
Creating RawArray with float64 data, n_channels=89, n_times=3122151
    Range : 0 ... 3122150 =      0.000 ...  1524.487 secs
Ready.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 55 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 55.00 Hz
- Upper transition bandwidth: 13.75 Hz (-6 dB cutoff frequency: 61.88 Hz)
- Filter length: 6759 samples (3.300 s)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.4s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.6s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.8s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  64 out of  64 | elapsed:   16.9s finished


Trigger channel TRIGGER has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)


C:\Users\USER\Documents\diplwmatikh\testakia\maiin\EEGPrep.py:57: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  raw.resample(128.0, npad = "auto")


...

Trigger channel TRIGGER has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)


C:\Users\USER\Documents\diplwmatikh\testakia\maiin\EEGPrep.py:57: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  raw.resample(128.0, npad = "auto")


Trigger channel TRIGGER has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)


C:\Users\USER\Documents\diplwmatikh\testakia\maiin\EEGPrep.py:57: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  raw.resample(128.0, npad = "auto")


Left turns: 112
Right turns: 128
Not setting metadata
240 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 240 events and 321 original time points ...
0 bad epochs dropped
  Added 240 EEG epochs from Copy of 09_27_2022_15_45_40-Exp_adadrive-Sbj_23-Ssn_03.dats.pkl
  Eye kept: L=240, R=240, M=240

Combining all files...
Done.
Saved EEG dataset to: E:\koorathota_data\pkl_data\pkl_data\pkl_data\eegdata\reve_eeg_dataset_with_eye.npz
Saved eye dataset to: E:\koorathota_data\pkl_data\pkl_data\pkl_data\eegdata\eye_dataset.npz
